# Private LB 1st Noise Ablation A/B GPU

GitHub upload notebook for the Private LB 1st noise-removal validation experiment.

Used files:
- `codex_mosquito_noise_removal_instructions.md`: criteria A/B definition
- This notebook: Private LB 1st model definitions + A/B/origin validation runner

Policy:
- Remove noise rows from the training fold only.
- Keep all validation rows for fair comparison.
- Do not remove test rows; generate quality flags only.

Default validation run:
- `n_each=1`
- `GRU/ODE epochs=8`
- `H epochs=3`

For the full Private LB recipe, change to `n_each=10`, `gru_ode_epochs=55`, `h_epochs=12`.


## 1. Check Environment


In [ ]:
from pathlib import Path
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

required = [
    Path("codex_mosquito_noise_removal_instructions.md"),
    Path("Data/train"),
    Path("Data/test"),
    Path("Data/train_labels.csv"),
]
for path in required:
    print(f"{path}: {path.exists()}")


## 2. Private LB 1st Model Definitions

The following cells are copied from `[Private_LB 1st] 코드 공유.ipynb`.


### Original Private LB 1st cell 4


In [ ]:
import os, glob, random, time
from datetime import datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler   # WeightedRandomSampler: H 학습의 θ-가중 오버샘플

# ── 토글 (실행 환경/방식에 맞춰 바꾸는 스위치) ──────────────────────────────
DATA_DIR     = os.environ.get('DATA_DIR', './Data')   # DACON 원본 위치 (Data/train, Data/test, train_labels.csv). 환경변수로도 지정 가능
FROM_SCRATCH = True            # True: 30모델 처음부터 학습(~2시간 10분, MPS) / False: ./models_goh30/*.pt 로드(~2분)
MODELS_DIR   = './models_goh30'                         # 학습 가중치 저장·로드 폴더
os.makedirs(MODELS_DIR, exist_ok=True)                 # 폴더 없으면 생성

# 연산 디바이스 자동 선택: CUDA > MPS(Apple Silicon) > CPU
DEVICE = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu'))

# ── 상수 (우승 레시피 하이퍼파라미터 · 0.7035 재현 위해 고정 — 바꾸면 결과 달라짐) ──
DT = 0.04; PRED_DT = 0.08; CLIP_THR = 1.33     # 관측간격 40ms · 예측시점 +80ms · 센서 클리핑 속력기준(scalar clip_flag용)
SPEED_BINS = [0.0, 0.3, 0.6, 0.9, 1.2, np.inf]                       # scalar 피처의 속력 구간 one-hot(5칸) 경계
SIGMA = 0.02; RHIT_TAU = 0.0015; RHIT_W = 2.0; HW = 0.5; GW = 0.5     # GRU/ODE 손실: 가우시안폭 σ · R-Hit 날카로움 τ · R-Hit가중 · Huber/soft 가중
FLIP_PROB = 0.5; NOISE_STD = 0.02; Y_FLIP = [1, 4, 7, 10]; INTERIOR_E = [5, 6, 7, 8]   # 증강: y-flip 확률·입력노이즈 / Y_FLIP=좌우대칭 시 부호반전할 seq채널 / INTERIOR_E=내부전이 사전학습 지점 e
N_EACH = 10                                                          # 아키텍처당 시드 수 (GRU10 + ODE10 + H10 = 30모델)
GRU_ODE_EPOCHS = 55; H_EPOCHS = 12; EMA_DECAY = 0.9                  # 전체학습 에폭(GRU·ODE / HyperPhysics) · EMA 가중평균 감쇠율

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)         # random·numpy·torch 시드 동시 고정 → 재현성 확보

print('device:', DEVICE, '| FROM_SCRATCH:', FROM_SCRATCH, '| DATA_DIR:', DATA_DIR)

### Original Private LB 1st cell 6


In [ ]:
# ── raw CSV(11×xyz) 로드 ──
def load_sample(path):
    df = pd.read_csv(path)
    return df[['x', 'y', 'z']].to_numpy(dtype=np.float32)   # (11,3) 위치 배열

# ── yaw 회전행렬: xy 속도방향을 +x축에 정렬(z축 회전) → 방위 불변 ──
def yaw_rotation_matrix(velocity):
    vx, vy = float(velocity[0]), float(velocity[1])
    speed_xy = np.sqrt(vx ** 2 + vy ** 2)
    if speed_xy < 1e-6:   # 정지면 회전 없음(단위행렬)
        return np.eye(3, dtype=np.float32)
    cos_yaw, sin_yaw = vx / speed_xy, vy / speed_xy
    return np.array([[cos_yaw, sin_yaw, 0.0], [-sin_yaw, cos_yaw, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)   # z축 회전행렬 (3,3)

# ── 시퀀스 피처 (11,13): 회전프레임 위치/속도/가속/저크/각속도 ──
def extract_seq_features(smoothed_pos, smoothed_vel, rot):
    last_pos = smoothed_pos[-1]
    rel_pos = (smoothed_pos - last_pos) @ rot.T   # 마지막 위치 기준 상대위치 → 회전프레임
    vel_rot = smoothed_vel @ rot.T   # 속도도 회전프레임 (11,3)
    accel = np.zeros_like(vel_rot)
    accel[1:-1] = (vel_rot[2:] - vel_rot[:-2]) / (2 * DT); accel[0] = accel[1]; accel[-1] = accel[-2]   # 가속도=속도 중앙차분(양끝 복제)
    jerk = np.zeros_like(accel)
    jerk[1:-1] = (accel[2:] - accel[:-2]) / (2 * DT); jerk[0] = jerk[1]; jerk[-1] = jerk[-2]   # 저크=가속도 중앙차분
    speed = np.linalg.norm(vel_rot, axis=1, keepdims=True)
    v_norm = vel_rot / (speed + 1e-12)
    cos_sim = (v_norm[:-1] * v_norm[1:]).sum(axis=1)
    angular_vel = np.concatenate([[cos_sim[0]], cos_sim])   # 연속 속도방향 코사인유사도(선회 정도)
    features = np.concatenate([rel_pos, vel_rot, accel, jerk, angular_vel[:, None]], axis=1)   # 3+3+3+3+1=13채널
    return features.astype(np.float32)

# ── 스칼라 피처: 궤적 요약 통계 ──
def extract_scalar_features(smoothed_pos, smoothed_vel):
    speeds = np.linalg.norm(smoothed_vel, axis=1); last_speed = float(speeds[-1])
    vel_diff = np.diff(smoothed_vel, axis=0) / DT; accel_mag = np.linalg.norm(vel_diff, axis=1)
    last_accel = float(accel_mag[-1]); mean_accel = float(accel_mag.mean())
    t = np.arange(len(smoothed_pos), dtype=np.float32); r2_list = []
    for dim in range(3):
        y = smoothed_pos[:, dim]; coeffs = np.polyfit(t, y, 1); y_pred = np.polyval(coeffs, t)
        ss_res = float(np.sum((y - y_pred) ** 2)); ss_tot = float(np.sum((y - y.mean()) ** 2))
        r2_list.append(1.0 - ss_res / (ss_tot + 1e-10))
    linearity = float(np.mean(r2_list)); clip_flag = float(last_speed > CLIP_THR)   # 직선성(R²) · 클리핑여부(속력>1.33)
    v_norm = smoothed_vel / (np.linalg.norm(smoothed_vel, axis=1, keepdims=True) + 1e-12)
    cos_sim_all = (v_norm[:-1] * v_norm[1:]).sum(axis=1)
    dir_consistency = float(cos_sim_all.mean()); delta_speed = float(speeds[-1] - speeds[-2])
    last_dir_change = float(cos_sim_all[-1])
    last_vel_norm = v_norm[-1]; last_accel_vec = vel_diff[-1]
    tangential = np.dot(last_accel_vec, last_vel_norm) * last_vel_norm
    last_normal_accel = float(np.linalg.norm(last_accel_vec - tangential))   # 법선(구심)가속 = 선회 강도
    speed_bin = np.zeros(5, dtype=np.float32)
    for k in range(5):   # 마지막 속력 → 5구간 one-hot
        if SPEED_BINS[k] <= last_speed < SPEED_BINS[k + 1]: speed_bin[k] = 1.0; break
    scalar = np.array([last_speed, last_accel, mean_accel, linearity, clip_flag,
                       dir_consistency, delta_speed, last_dir_change, last_normal_accel], dtype=np.float32)
    return np.concatenate([scalar, speed_bin])   # 9 + 5 = 14

# ── 가변길이 윈도우(L≥4) → 피처 (build_features_clean의 일반화) ──
def window_features(W):
    '''길이 L>=4 윈도우 → seq(L,13), scalar(22), rot, base, last_pos. (build_features_clean의 일반화)'''
    W = W.astype(np.float64); vel = np.gradient(W, DT, axis=0); rot = yaw_rotation_matrix(vel[-1])   # 속도=중앙차분, 회전=마지막 속도방향
    seq = extract_seq_features(W, vel, rot); b14 = extract_scalar_features(W, vel)
    sp = np.linalg.norm(vel, axis=1); steps = np.linalg.norm(np.diff(W, axis=0), axis=1); L = len(W)
    path = float(steps.sum()); net = float(np.linalg.norm(W[-1] - W[0])); straight = net / (path + 1e-8)
    t = np.arange(float(L)); noise = float(np.mean([(W[:, d] - np.polyval(np.polyfit(t, W[:, d], 2), t)).std() for d in range(3)]))
    k = min(4, L); acc_trend = float(np.polyfit(np.arange(float(k)), sp[-k:], 1)[0])
    sc = np.concatenate([b14, [float(sp.max()), float(sp.std()), float(sp[-3:].mean()), float(sp[-5:].mean()),   # 추가 8: 최대·표준편차·최근3/5평균 속력 등
                               path, straight, noise, acc_trend]]).astype(np.float32)
    base = (W[-1] + 2.0 * (W[-1] - W[-2])).astype(np.float32)   # base = cv_1step (+80ms 등속)
    return seq.astype(np.float32), sc, rot.astype(np.float32), base, W[-1].astype(np.float32)   # seq, scalar22, rot, base, last_pos

# ── L=11 전용 진입점(테스트 전처리) ──
def build_features_clean(X):
    '''L=11 전용 (window_features와 동일 산식). 테스트 전처리용.'''
    return window_features(X)

# ── norm_stats로 표준화 ──
def normalize(seq, scalar, stats):
    seq_n = ((seq - stats['seq_mean']) / stats['seq_std']).astype(np.float32)
    scal_n = ((scalar - stats['scalar_mean']) / stats['scalar_std']).astype(np.float32)
    return seq_n, scal_n

print('피처 함수 정의 완료')

### Original Private LB 1st cell 11


In [ ]:
class AttnGRU(nn.Module):   # phaseG_full
    def __init__(self, seq_dim=13, scal_dim=22, h=128, nl=3, dr=0.15):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(seq_dim, h), nn.LayerNorm(h))   # 13→128 입력 투영
        self.gru = nn.GRU(h, h, nl, batch_first=True, bidirectional=True, dropout=dr if nl > 1 else 0)   # 양방향 GRU(128,3층)→256
        self.attn = nn.Linear(h*2, 1)   # attention 점수
        self.head = nn.Sequential(nn.Linear(h*6+scal_dim, 256), nn.GELU(), nn.Dropout(dr),   # 풀링3(768)+scalar22 → 3D 잔차
                                  nn.Linear(256, 64), nn.GELU(), nn.Linear(64, 3))
    def forward(self, seq, scal, mask):
        x = self.proj(seq); out, _ = self.gru(x); last = out[:, -1, :]; m = mask.unsqueeze(-1)   # last=마지막 스텝
        mean = (out*m).sum(1)/m.sum(1).clamp(min=1)   # mask 평균 풀링(pad 제외)
        score = self.attn(out).squeeze(-1).masked_fill(mask < 0.5, -1e9)   # pad는 softmax에서 제외
        att = (torch.softmax(score, dim=1).unsqueeze(-1)*out).sum(1)   # attention 가중 풀링
        return self.head(torch.cat([last, mean, att, scal], -1))   # 3풀링+scalar → 잔차

### Original Private LB 1st cell 13


In [ ]:
class ODEModel(nn.Module):   # phaseODE_full (train_phaseODE의 MaskedBiGRU와 동일)
    def __init__(self, seq_dim=13, scal_dim=22, h=128, nl=2, dr=0.15, latent=96, nsteps=4):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(seq_dim, h), nn.LayerNorm(h))
        self.gru = nn.GRU(h, h, nl, batch_first=True, bidirectional=True, dropout=dr if nl > 1 else 0)
        self.to_latent = nn.Sequential(nn.Linear(h*4+scal_dim, latent), nn.LayerNorm(latent), nn.GELU())   # GRU표현(256)+scalar → latent 96
        self.accel = nn.Sequential(nn.Linear(3+3+latent, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dr),   # 가속도장 NN: (pos3,vel3,latent)→acc3
                                   nn.Linear(128, 64), nn.GELU(), nn.Linear(64, 3))
        self.damping = nn.Parameter(torch.tensor([1.0, 1.0, 1.0]))   # 축별 댐핑(학습 파라미터)
        self.bias = nn.Parameter(torch.zeros(3))
        self.nsteps = nsteps; self.dt = 0.08/nsteps   # +80ms를 4스텝으로 적분
    # 미분방정식 우변(댐핑 가속도장)
    def _deriv(self, rpos, rvel, lat):
        a = self.accel(torch.cat([rpos, rvel, lat], -1))
        return rvel, -self.damping*rvel+a   # dx/dt=v, dv/dt=a_NN−damping·v
    def forward(self, seq, scal, mask):
        x = self.proj(seq); out, _ = self.gru(x); m = mask.unsqueeze(-1)
        mean = (out*m).sum(1)/m.sum(1).clamp(min=1)
        lat = self.to_latent(torch.cat([out[:, -1, :], mean, scal], -1))   # 시퀀스→latent
        rpos = torch.zeros(seq.size(0), 3, device=seq.device); rvel = torch.zeros_like(rpos)   # 잔차 위치/속도 0에서 시작
        for _ in range(self.nsteps):   # RK4 4스텝 적분
            dt = self.dt
            dp1, dv1 = self._deriv(rpos, rvel, lat)
            dp2, dv2 = self._deriv(rpos+0.5*dt*dp1, rvel+0.5*dt*dv1, lat)
            dp3, dv3 = self._deriv(rpos+0.5*dt*dp2, rvel+0.5*dt*dv2, lat)
            dp4, dv4 = self._deriv(rpos+dt*dp3, rvel+dt*dv3, lat)
            rpos = rpos+(dt/6)*(dp1+2*dp2+2*dp3+dp4)
            rvel = rvel+(dt/6)*(dv1+2*dv2+2*dv3+dv4)
        return rpos+self.bias   # 적분 변위(잔차)+bias

### Original Private LB 1st cell 15


In [ ]:
# ── 학습 데이터셋: 가변 윈도우 + 확장 타깃 + θ-가중 오버샘플 ──
class SlidingWindowDataset(Dataset):
    def __init__(self, X, y, min_win=3, mode="extended", device="cpu"):
        X_tensor = torch.tensor(X, dtype=torch.float32); y_tensor = torch.tensor(y, dtype=torch.float32)
        windows = []
        for i in range(len(X)):
            targets = [4, 5, 6, 7, 8, 9, 10, 12] if mode == "extended" else [12, 10]   # 내부 지점들도 타깃(확장)
            for target_idx in targets:
                end_idx = target_idx - 2
                max_w = end_idx + 2 if mode == "extended" else (12 if target_idx == 12 else 10)
                for w in range(min_win, max_w):
                    windows.append((i, w, target_idx))
        X_list = []; y_list = []
        for i, w, target_idx in windows:
            X_orig = X_tensor[i]; end_idx = target_idx - 2
            pts = X_orig[end_idx - w + 1: end_idx + 1]
            target = y_tensor[i] if target_idx == 12 else X_orig[target_idx]
            if w < 11:
                v0 = pts[1] - pts[0]; n_pad = 11 - w
                js = torch.arange(n_pad, 0, -1, dtype=torch.float32)
                pad = pts[0:1] - js.unsqueeze(1) * v0.unsqueeze(0)
                X_padded = torch.cat([pad, pts], dim=0)
            else:
                X_padded = pts.clone()
            X_list.append(X_padded); y_list.append(target)
        self.X_all = torch.stack(X_list).to(device); self.y_all = torch.stack(y_list).to(device)
        diffs = self.X_all[:, 1:] - self.X_all[:, :-1]
        n1 = diffs[:, 1:].norm(dim=2).clamp(min=1e-8); n2 = diffs[:, :-1].norm(dim=2).clamp(min=1e-8)
        cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2) / (n1 * n2)).clamp(-1, 1)
        theta_last = torch.acos(cos_t[:, -1])
        self.theta_weights = (1.0 + 4.0 * (theta_last / 1.0).clamp(0, 1)).cpu().numpy()   # 급선회(θ↑)일수록 최대 5× 가중

    def __len__(self): return len(self.X_all)
    def __getitem__(self, idx): return self.X_all[idx], self.y_all[idx]


# ───────────────────────── 원본 cell[5]: 피처/손실/블록 ─────────────────────────
# ── EMA로 로컬 속도(vl)·가속(al) 평활 ──
def _ema_va_local(diffs_local, alpha, beta):
    B, T, _ = diffs_local.shape
    one_m_a = 1.0 - alpha; one_m_b = 1.0 - beta
    vs = diffs_local.new_empty(B, T, 3); v = diffs_local[:, 0]; vs[:, 0] = v
    for t in range(1, T):
        v = alpha * diffs_local[:, t] + one_m_a * v; vs[:, t] = v
    vl = vs[:, -1]
    ad = vs[:, 1:] - vs[:, :-1]; a = ad[:, 0]
    for t in range(1, T - 1):
        a = beta * ad[:, t] + one_m_b * a
    return vl, a


# ── soft R-Hit 손실: 1.3cm 임계 sigmoid 근사 ──
def _soft_hit_loss(pred, target, thr=0.013012, k=408.348):
    return (1 - torch.sigmoid(-(torch.norm(pred - target, dim=1) - thr) * k)).mean()


# ── 24차원 물리 피처 + 로컬 프레임 R + diffs 추출 ──
def extract_features(X, mean_stats=None, std_stats=None, dir_net=None, heading_mode="3step"):
    device = X.device
    p_last = X[:, 10]; diffs = X[:, 1:] - X[:, :-1]
    n1 = diffs[:, 1:].norm(dim=2, keepdim=True) + 1e-8; n2 = diffs[:, :-1].norm(dim=2, keepdim=True) + 1e-8
    cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2, keepdim=True) / (n1 * n2)).clamp(-1, 1)
    theta_seq = torch.acos(cos_t).squeeze(2)
    theta = theta_seq[:, -1:]; theta_mean = theta_seq.mean(1, keepdim=True); theta_std = theta_seq.std(1, keepdim=True)
    theta_vel = theta_seq[:, -1:] - theta_seq[:, -2:-1]
    theta_acc = theta_seq[:, -1:] - 2 * theta_seq[:, -2:-1] + theta_seq[:, -3:-2]
    theta_trend = theta_seq[:, -1:] - theta_seq[:, -3:].mean(1, keepdim=True)
    if dir_net is not None:
        speed_seq = diffs.norm(dim=2); state = torch.cat([speed_seq, theta_seq], dim=1)
        if dir_net[0].in_features == 29:
            z_speed_seq = diffs[:, :, 2].abs(); state = torch.cat([state, z_speed_seq], dim=1)
        weights = F.softmax(dir_net(state), dim=1); v_sm = (diffs * weights.unsqueeze(2)).sum(dim=1)
    else:
        v_sm = (3 * diffs[:, -1] + 2 * diffs[:, -2] + diffs[:, -3]) / 6.0 if heading_mode == "3step" else diffs[:, -1]
    fwd = v_sm / (v_sm.norm(dim=1, keepdim=True) + 1e-8)
    up_w = torch.zeros_like(fwd); up_w[:, 2] = 1.0
    up_w[fwd[:, 2].abs() > 0.99] = torch.tensor([0., 1., 0.], device=device)
    right = torch.cross(fwd, up_w, dim=1); right = right / (right.norm(dim=1, keepdim=True) + 1e-8)
    up = torch.cross(right, fwd, dim=1); up = up / (up.norm(dim=1, keepdim=True) + 1e-8)
    R = torch.stack([fwd, right, up], dim=2)   # 로컬 프레임(forward,right,up)
    v_last = diffs[:, -1]; v_prev1 = diffs[:, -2]; speed = v_last.norm(dim=1, keepdim=True)
    a_last = v_last - v_prev1; acc_mag = a_last.norm(dim=1, keepdim=True)
    v_local = torch.matmul(v_last.unsqueeze(1), R).squeeze(1)
    a_local = torch.matmul(a_last.unsqueeze(1), R).squeeze(1)
    X_local = torch.matmul(X - p_last.unsqueeze(1), R); p_std_local = X_local.std(1)
    v_local_abs = v_local.abs()
    jerk_g = diffs[:, -1] - 2 * diffs[:, -2] + diffs[:, -3]
    jerk_l = torch.matmul(jerk_g.unsqueeze(1), R).squeeze(1); jerk_mag = jerk_g.norm(dim=1, keepdim=True)
    features = torch.cat([v_local, a_local, speed, acc_mag, theta, theta_mean, theta_std, theta_trend,   # 24차원 물리 피처
                          theta_vel, theta_acc, p_std_local, v_local_abs, jerk_l, jerk_mag], dim=1)
    if mean_stats is None or std_stats is None:
        mean_stats = features.mean(0, keepdim=True); std_stats = features.std(0, keepdim=True) + 1e-8
    return (features - mean_stats) / std_stats, diffs, p_last, theta, theta_mean, theta_std, theta_seq, R, speed, mean_stats, std_stats


# ── 잔차 블록(LayerNorm+GELU) ──
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.15), nn.Linear(dim, dim))
        self.ln = nn.LayerNorm(dim)
    def forward(self, x): return self.ln(x + self.net(x))


# ── 마지막 층 0 + prior_bias에서 출발(물리 prior) ──
class PriorBiasedLinear(nn.Module):
    def __init__(self, in_features, out_features, prior_bias):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.register_buffer('prior_bias', prior_bias.clone().detach())
        with torch.no_grad():
            nn.init.zeros_(self.linear.weight); nn.init.zeros_(self.linear.bias)
    def forward(self, x): return self.linear(x) + self.prior_bias


# ── Rodrigues 회전: v를 각속도 w로 회전(뱅킹 턴) ──
def rodrigues_rotate(v, w):
    theta = w.norm(dim=1, keepdim=True); k = w / (theta + 1e-8)
    cos_t = torch.cos(theta); sin_t = torch.sin(theta)
    dot = (v * k).sum(dim=1, keepdim=True); cross = torch.cross(k, v, dim=1)
    return v * cos_t + cross * sin_t + k * dot * (1.0 - cos_t)


# ───────────────────────── 원본 cell[7]: HyperPhysics_xy2 ─────────────────────────
# ── HyperPhysics 본체 (예측식은 위 마크다운 cell 4c 참고) ──
class HyperPhysics_xy2(nn.Module):
    def __init__(self, input_dim=24, **kwargs):
        super().__init__()
        self.sh_thr = kwargs.pop('sh_thr', 0.013012); self.sh_k = kwargs.pop('sh_k', 408.348044)
        self.mse_w = kwargs.pop('mse_w', 129.172037); self.local_w = kwargs.pop('local_w', 0.050941)
        self.theta_thr = kwargs.pop('theta_thr', 1.087618); self.speed_thr = kwargs.pop('speed_thr', 0.034583)   # 회전 게이팅 임계값(원본 튜닝 고정)
        self.lr = 0.005400; self.wd = 0.005659
        self.register_buffer("mean_stats", torch.zeros(1, input_dim)); self.register_buffer("std_stats", torch.ones(1, input_dim))
        prior_dir = torch.tensor([-10., -10., -10., -10., -10., -10., -10., 0., 0.693, 1.098])
        self.dir_net = nn.Sequential(nn.Linear(29, 24), nn.LayerNorm(24), nn.GELU(), PriorBiasedLinear(24, 10, prior_dir))   # heading 가중(3step 방향) 학습
        prior_ema = torch.zeros(6)
        self.temporal_net = nn.Sequential(nn.Linear(9, 32), nn.LayerNorm(32), nn.GELU(), PriorBiasedLinear(32, 6, prior_ema))   # EMA 계수(alpha,beta) 출력
        prior_dyn = torch.tensor([0., 0., 0., 0., 0., 0.] + [-4.] * 24)
        self.dynamics_net = nn.Sequential(nn.Linear(input_dim, 96), nn.LayerNorm(96), nn.GELU(), ResBlock(96), PriorBiasedLinear(96, 30, prior_dyn))   # w_v,w_a,exp_v,exp_a 동역학 계수
        self.omega_w = nn.Parameter(torch.tensor([0.0, -0.5, -1.0]))
        self.omega_net = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, 48), nn.GELU(), nn.Linear(48, 3))   # 각속도 보정(omega_delta)
        with torch.no_grad():
            nn.init.normal_(self.omega_net[-1].weight, std=0.01); nn.init.zeros_(self.omega_net[-1].bias)
        self.diffusion_net = nn.Sequential(nn.Linear(input_dim, 32), nn.LayerNorm(32), nn.GELU(), nn.Linear(32, 3))

    def get_features(self, X, mean_stats=None, std_stats=None):
        return extract_features(X, mean_stats, std_stats, self.dir_net, heading_mode="3step")

    @staticmethod
    def _rotation_vector(d_prev, d_curr):
        n_prev = d_prev.norm(dim=1, keepdim=True).clamp(min=1e-8); n_curr = d_curr.norm(dim=1, keepdim=True).clamp(min=1e-8)
        d_hat_prev = d_prev / n_prev; d_hat_curr = d_curr / n_curr
        cross = torch.linalg.cross(d_hat_prev, d_hat_curr, dim=1); sin_t = cross.norm(dim=1, keepdim=True).clamp(min=1e-8)
        cos_t = (d_hat_prev * d_hat_curr).sum(1, keepdim=True).clamp(-0.9999, 0.9999); theta = torch.atan2(sin_t, cos_t)
        speed_gate = torch.sigmoid((n_prev + n_curr) * 500 - 5)
        return cross / sin_t * theta * speed_gate

    def forward(self, features, diffs, p_last, theta, speed, R):
        B = diffs.shape[0]
        ema_raw = self.temporal_net(features[:, 8:17])
        alpha = torch.sigmoid(ema_raw[:, 0:3]) * 0.8 + 0.1; beta = torch.sigmoid(ema_raw[:, 3:6]) * 0.199 + 0.8   # EMA 계수 범위 제한
        dyn_raw = self.dynamics_net(features)
        w_v = 2.0 + dyn_raw[:, 0:3]; w_a = 1.0 + dyn_raw[:, 3:6]   # 속도/가속 기여 가중(prior 2.0/1.0)
        v_local_abs = features[:, 17:20]; v_local_abs2 = v_local_abs * v_local_abs; theta2 = theta * theta
        exp_v = (F.softplus(dyn_raw[:, 6:9]) * v_local_abs + F.softplus(dyn_raw[:, 9:12]) * v_local_abs2 +
                 F.softplus(dyn_raw[:, 12:15]) * theta + F.softplus(dyn_raw[:, 15:18]) * theta2)
        exp_a = (F.softplus(dyn_raw[:, 18:21]) * v_local_abs + F.softplus(dyn_raw[:, 21:24]) * v_local_abs2 +
                 F.softplus(dyn_raw[:, 24:27]) * theta + F.softplus(dyn_raw[:, 27:30]) * theta2)
        diffs_local = torch.matmul(diffs, R)   # 변위들을 로컬 프레임으로
        vl, al = _ema_va_local(diffs_local, alpha, beta)   # EMA 평활 속도/가속
        diff_speed = diffs_local.norm(dim=2)
        def rv_masked(ka, kb):
            rv = self._rotation_vector(diffs_local[:, ka], diffs_local[:, kb])
            valid = ((diff_speed[:, ka] > 1e-5) & (diff_speed[:, kb] > 1e-5)).float()
            return rv * valid.unsqueeze(1), valid
        ov1, vm1 = rv_masked(-2, -1); ov2, vm2 = rv_masked(-3, -2); ov3, vm3 = rv_masked(-4, -3)
        w_logits = self.omega_w.view(1, 3).expand(B, -1)
        masks = torch.stack([vm1, vm2, vm3], dim=1)
        w_attn = F.softmax(w_logits.masked_fill(masks == 0, -1e9), dim=1)
        omega_hist = (w_attn[:, 0].unsqueeze(1) * ov1 + w_attn[:, 1].unsqueeze(1) * ov2 + w_attn[:, 2].unsqueeze(1) * ov3)
        current_speed = speed.view(B, 1) if speed is not None else diff_speed[:, -1].unsqueeze(1)
        omega_speed_gate = torch.sigmoid(current_speed * 500 - 5)
        omega_delta = self.omega_net(features) * omega_speed_gate
        theta_scalar = theta.view(B, 1)
        theta_gate = torch.sigmoid((theta_scalar - self.theta_thr) * 10)   # θ가 임계 넘어야 회전 ON
        speed_gate_strong = torch.sigmoid((current_speed - self.speed_thr) * 200)   # 속력 임계 게이트
        rotation_gate = theta_gate * speed_gate_strong
        omega = (omega_hist + omega_delta) * rotation_gate   # 최종 각속도(게이팅 적용)
        v_rotated = rodrigues_rotate(vl, omega)   # 속도벡터를 ω로 회전 = 뱅킹 턴
        pred_local = (w_v * torch.exp(-exp_v)) * v_rotated + (w_a * torch.exp(-exp_a)) * al   # 로컬 잔차 = 감쇠(회전속도)+감쇠(가속)
        log_var = self.diffusion_net(features).clamp(min=-5.0, max=5.0)
        pred_global = p_last + torch.einsum('nij,nj->ni', R, pred_local)   # 글로벌: last + R·pred_local
        return pred_global, pred_local, log_var

    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None, **kwargs):
        sh = _soft_hit_loss(pp, yr, thr=self.sh_thr, k=self.sh_k)
        loss = sh + self.mse_w * F.mse_loss(pp, yr)   # soft-hit + MSE
        if pred_local is not None and yr_local is not None and log_var is not None:
            squared_error = (pred_local - yr_local) ** 2
            nll_loss = 0.5 * (torch.exp(-log_var) * squared_error + log_var)   # 로컬 잔차 NLL(불확실성 가중)
            loss = loss + self.local_w * nll_loss.mean()
        return loss


# ───────────────────────── 우리 fold OOF 래퍼 ─────────────────────────
# ── (참고용) OOF 검증 래퍼 — GOH30 재현 경로엔 미사용 ──
def load_train():
    paths = sorted((ROOT/'Data'/'train').glob('*.csv'))
    labs = pd.read_csv(ROOT/'Data'/'train_labels.csv').sort_values('id').reset_index(drop=True)[['x', 'y', 'z']].to_numpy(np.float32)
    X = np.stack([pd.read_csv(p)[['x', 'y', 'z']].to_numpy(np.float32) for p in paths])
    return X, labs


# ── 평가지표 R-Hit@1cm ──
def r_hit(p, t, thr=0.01): return float(np.mean(np.linalg.norm(p - t, axis=1) <= thr))

### Original Private LB 1st cell 17


In [ ]:
# ── GRU/ODE 손실: Huber + 가우시안-soft + Soft R-Hit ──
def combined_loss(pred,true):
    d=0.01; hub=F.huber_loss(pred,true,delta=d)/(0.5*d*d)   # ① Huber(1cm) 정규화
    d2=(pred-true).pow(2).sum(-1); soft=(1-torch.exp(-d2/(2*SIGMA**2))).mean()   # ② 가우시안-soft(σ=0.02)
    dd=torch.sqrt(d2+1e-12); sr=-torch.sigmoid((0.01-dd)/RHIT_TAU).mean()   # ③ Soft R-Hit: 1cm 적중 미분근사
    return HW*hub+GW*soft+RHIT_W*sr   # 0.5·Huber + 0.5·soft + 2.0·R-Hit

# ── 평가지표 R-Hit@1cm ──
def r_hit(p, t, thr=0.01):
    return float(np.mean(np.linalg.norm(p - t, axis=1) <= thr))

### Original Private LB 1st cell 18


In [ ]:
# ── GRU/ODE 학습(캐시) → EMA 가중치 반환 ──
def train_cache_seed(seed, factory):
    '''GRU/ODE 전체데이터 학습 (캐시 사용) → EMA state_dict 반환.'''
    dev = DEVICE
    seq = torch.tensor(CACHE['seq']); scal = torch.tensor(CACHE['scal'])
    msk = torch.tensor(CACHE['mask']); tgt = torch.tensor(CACHE['tgt'])
    N = len(seq); idx_all = np.arange(N)
    torch.manual_seed(1000 + seed); np.random.seed(1000 + seed)   # 시드별 재현 고정
    model = factory().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)   # AdamW (lr 2e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=GRU_ODE_EPOCHS)   # Cosine LR (55ep)
    flip = torch.tensor(Y_FLIP, device=dev)
    ema = {k: v.detach().clone() for k, v in model.state_dict().items()}
    for ep in range(1, GRU_ODE_EPOCHS + 1):
        model.train(); np.random.shuffle(idx_all)
        for i in range(0, N, 256):
            b = idx_all[i:i + 256]
            s = seq[b].to(dev); c = scal[b].to(dev); mk = msk[b].to(dev); tg = tgt[b].to(dev)
            if torch.rand(1).item() < FLIP_PROB:   # 50% 확률 y-flip 증강(좌우대칭)
                s = s.clone(); s[:, :, flip] *= -1; tg = tg.clone(); tg[:, 1] *= -1   # seq y채널 + 타깃 y 부호반전
            s = s + torch.randn_like(s) * NOISE_STD * mk.unsqueeze(-1)   # 입력 노이즈 증강(유효 스텝만)
            opt.zero_grad(); loss = combined_loss(model(s, c, mk), tg); loss.backward()   # 잔차 예측 → combined_loss
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5); opt.step()   # grad clip 0.5
        sch.step()
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point: ema[k].mul_(EMA_DECAY).add_(v, alpha=1 - EMA_DECAY)
                else: ema[k] = v.detach().clone()
    return ema

# ── HyperPhysics 학습(raw 궤적) → EMA 가중치 반환 ──
def train_h_seed(seed, X, Y):
    '''HyperPhysics 전체데이터 학습 (raw 궤적 사용) → EMA state_dict 반환.'''
    dev = DEVICE; set_seed(1000 + seed)
    ds = SlidingWindowDataset(X, Y, min_win=3, mode="extended", device=dev)   # 가변윈도우 데이터셋
    loader = DataLoader(ds, batch_size=256, sampler=WeightedRandomSampler(ds.theta_weights, len(ds), replacement=True))   # θ-가중 오버샘플(급선회 강조)
    model = HyperPhysics_xy2().to(dev)
    with torch.no_grad():
        *_, mn, st = model.get_features(torch.tensor(X, dtype=torch.float32, device=dev))
        model.mean_stats.copy_(mn); model.std_stats.copy_(st)   # 피처 표준화 통계 주입
    opt = torch.optim.AdamW(model.parameters(), lr=model.lr, weight_decay=model.wd)   # AdamW (lr=model.lr 0.0054)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=4, gamma=0.6)   # StepLR (4ep마다 ×0.6)
    ema = {k: v.detach().clone() for k, v in model.state_dict().items()}
    for ep in range(1, H_EPOCHS + 1):
        model.train()
        for Xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            ft, df, pl, th, _, _, _, Rt, sp, _, _ = model.get_features(Xb, model.mean_stats, model.std_stats)   # 물리 피처/프레임 추출
            pp, pred_local, log_var = model(ft, df, pl, th, sp, Rt)
            yr_local = torch.matmul((yb - pl).unsqueeze(1), Rt).squeeze(1)   # 타깃 잔차를 로컬 프레임으로(NLL용)
            loss = model.compute_loss(pp, yb, pred_local, yr_local, log_var)   # soft-hit+MSE+NLL
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()   # grad clip 1.0
        sch.step()
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if v.dtype.is_floating_point: ema[k].mul_(EMA_DECAY).add_(v, alpha=1 - EMA_DECAY)
                else: ema[k] = v.detach().clone()
    return ema

print('학습 함수 정의 완료')

## 3. Noise Ablation Runner

This is the executable A/B/origin comparison code. It removes noise only from the training fold.


In [ ]:
from __future__ import annotations

import argparse
import csv
import glob
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


R_HIT = 0.01


def stable_fold_id(sample_id: str, folds: int) -> int:
    digest = hashlib.md5(sample_id.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % folds


def metrics(pred: np.ndarray, true: np.ndarray) -> dict[str, float | int]:
    err = np.linalg.norm(pred - true, axis=1)
    return {
        "hit": float(np.mean(err <= R_HIT)),
        "hits": int(np.sum(err <= R_HIT)),
        "mean": float(np.mean(err)),
        "median": float(np.quantile(err, 0.50)),
        "q75": float(np.quantile(err, 0.75)),
        "q90": float(np.quantile(err, 0.90)),
        "q95": float(np.quantile(err, 0.95)),
    }


def private_notebook_path() -> Path:
    matches = sorted(Path(".").glob("*Private_LB*.ipynb"))
    if not matches:
        raise FileNotFoundError("Could not find [Private_LB 1st] notebook")
    return matches[0]


def load_private_namespace() -> dict[str, object]:
    path = private_notebook_path()
    nb = json.loads(path.read_text(encoding="utf-8"))
    ns: dict[str, object] = {"__name__": "__private_lb_ablation__"}
    # Execute only definition/config cells. Skip pip install, preprocessing,
    # full training, and submission cells.
    for idx in [4, 6, 11, 13, 15, 17, 18]:
        code = "".join(nb["cells"][idx]["source"])
        exec(compile(code, f"{path}:cell_{idx}", "exec"), ns)
    return ns


def quality_features(ids: list[str], trajs: np.ndarray, targets: np.ndarray | None) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    dt = 0.04
    for i, sample_id in enumerate(ids):
        pos = np.asarray(trajs[i], dtype=np.float32)
        dpos = np.diff(pos, axis=0)
        step_dist = np.linalg.norm(dpos, axis=1)
        vel = dpos / dt
        acc = np.diff(vel, axis=0) / dt
        jerk = np.diff(acc, axis=0) / dt
        median_step = float(np.median(step_dist))
        interp_resids = []
        for t in range(1, 10):
            expected = 0.5 * (pos[t - 1] + pos[t + 1])
            interp_resids.append(float(np.linalg.norm(pos[t] - expected)))
        max_interp_resid = float(max(interp_resids))
        displacement = float(np.linalg.norm(pos[-1] - pos[0]))
        path_length = float(np.sum(step_dist))
        row: dict[str, object] = {
            "id": sample_id,
            "min_step": float(step_dist.min()),
            "max_step": float(step_dist.max()),
            "median_step": median_step,
            "max_step_over_median": float(step_dist.max() / (median_step + 1e-8)),
            "zero_step_count": int(np.sum(step_dist == 0)),
            "tiny_step_lt_1e4": bool(float(step_dist.min()) < 1e-4),
            "max_acc": float(np.linalg.norm(acc, axis=1).max()),
            "max_jerk": float(np.linalg.norm(jerk, axis=1).max()),
            "max_interp_resid": max_interp_resid,
            "max_interp_resid_over_med_step": float(max_interp_resid / (median_step + 1e-8)),
            "straightness": float(displacement / (path_length + 1e-8)),
        }
        if targets is not None:
            target = np.asarray(targets[i], dtype=np.float32)
            last_pos = pos[-1]
            last_vel = (pos[-1] - pos[-2]) / dt
            cv80_pred = last_pos + 0.08 * last_vel
            future_vel = (target - last_pos) / 0.08
            row.update(
                {
                    "target_dist_from_last": float(np.linalg.norm(target - last_pos)),
                    "target_minus_cv80": float(np.linalg.norm(target - cv80_pred)),
                    "future_v_change_norm": float(np.linalg.norm(future_vel - last_vel)),
                }
            )
        rows.append(row)
    return pd.DataFrame(rows)


def apply_criteria_a_conservative(q: pd.DataFrame, has_target: bool) -> pd.Series:
    mask = (
        (q["zero_step_count"] > 0)
        | (q["min_step"] < 1e-4)
        | (q["max_step"] > 0.05398218)
        | (q["max_acc"] > 54.68043)
        | (q["max_jerk"] > 1528.03563)
        | (q["max_interp_resid"] > 0.04374434)
    )
    if has_target:
        mask = mask | (
            (q["target_minus_cv80"] > 0.14535968)
            | (q["future_v_change_norm"] > 1.81699603)
        )
    return mask.fillna(False).astype(bool)


def apply_criteria_b_strict(q: pd.DataFrame, has_target: bool) -> pd.Series:
    return (
        apply_criteria_a_conservative(q, has_target=has_target)
        | (q["max_interp_resid_over_med_step"] > 1.5)
        | (q["max_step_over_median"] > 3.0)
    ).fillna(False).astype(bool)


def build_noise_reports(
    data_dir: Path,
    out_dir: Path,
    ids: list[str],
    x_all: np.ndarray,
    y_all: np.ndarray,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    out_dir.mkdir(parents=True, exist_ok=True)
    train_quality = quality_features(ids, x_all, y_all)
    train_quality["remove_criteria_a_conservative"] = apply_criteria_a_conservative(
        train_quality, has_target=True
    ).to_numpy()
    train_quality["remove_criteria_b_strict"] = apply_criteria_b_strict(
        train_quality, has_target=True
    ).to_numpy()
    train_quality.to_csv(out_dir / "train_quality_report_all_criteria.csv", index=False)
    for mode in ["criteria_a_conservative", "criteria_b_strict"]:
        flag = f"remove_{mode}"
        removed = train_quality.loc[train_quality[flag], ["id"]]
        removed.to_csv(out_dir / f"removed_train_ids_{mode}.csv", index=False)
        keep = ~train_quality[flag].to_numpy(dtype=bool)
        pd.DataFrame(
            {
                "id": np.asarray(ids, dtype=object)[keep],
                "x": y_all[keep, 0],
                "y": y_all[keep, 1],
                "z": y_all[keep, 2],
            }
        ).to_csv(out_dir / f"train_cleaned_{mode}.csv", index=False)

    test_paths = sorted(glob.glob(str(data_dir / "test" / "*.csv")))
    test_ids = [Path(p).stem for p in test_paths]
    test_x = np.stack([pd.read_csv(p)[["x", "y", "z"]].to_numpy(np.float32) for p in test_paths])
    test_quality = quality_features(test_ids, test_x, None)
    test_quality["remove_criteria_a_conservative"] = apply_criteria_a_conservative(
        test_quality, has_target=False
    ).to_numpy()
    test_quality["remove_criteria_b_strict"] = apply_criteria_b_strict(
        test_quality, has_target=False
    ).to_numpy()
    test_quality.to_csv(out_dir / "test_quality_report_with_noise_flags.csv", index=False)
    return (
        train_quality,
        train_quality["remove_criteria_a_conservative"].to_numpy(dtype=bool),
        train_quality["remove_criteria_b_strict"].to_numpy(dtype=bool),
    )


def build_stats_and_cache(ns: dict[str, object], train_paths: list[str], labels: np.ndarray) -> dict[str, object]:
    build_features_clean = ns["build_features_clean"]
    window_features = ns["window_features"]
    inter_e = ns["INTERIOR_E"]
    seqs_for_stats, scals_for_stats = [], []
    for p in train_paths:
        x = pd.read_csv(p)[["x", "y", "z"]].to_numpy()
        seq, scal, *_ = build_features_clean(x)
        seqs_for_stats.append(seq)
        scals_for_stats.append(scal)
    seqs_for_stats = np.stack(seqs_for_stats)
    scals_for_stats = np.stack(scals_for_stats)
    stats = {
        "seq_mean": seqs_for_stats.reshape(-1, 13).mean(0),
        "seq_std": seqs_for_stats.reshape(-1, 13).std(0) + 1e-8,
        "scalar_mean": scals_for_stats.mean(0),
        "scalar_std": scals_for_stats.std(0) + 1e-8,
    }

    seq, scal, mask, tgt, rot, spd, traj, real = [], [], [], [], [], [], [], []
    for i, p in enumerate(train_paths):
        x = ns["load_sample"](p).astype(np.float64)
        examples = [(10, labels[i])] + [(e, x[e + 2]) for e in inter_e]
        for e, target in examples:
            w = x[: e + 1]
            length = len(w)
            s, sc, r, base, _ = window_features(w)
            s_n = ((s - stats["seq_mean"]) / stats["seq_std"]).astype(np.float32)
            sc_n = ((sc - stats["scalar_mean"]) / stats["scalar_std"]).astype(np.float32)
            pad = 11 - length
            seq11 = np.zeros((11, 13), np.float32)
            seq11[pad:] = s_n
            m = np.zeros(11, np.float32)
            m[pad:] = 1.0
            target_rot = (r @ (target - base)).astype(np.float32)
            seq.append(seq11)
            scal.append(sc_n)
            mask.append(m)
            tgt.append(target_rot)
            rot.append(r)
            spd.append(float(np.linalg.norm(np.gradient(w, ns["DT"], axis=0)[-1])))
            traj.append(i)
            real.append(int(e == 10))
    return {
        "STATS": stats,
        "CACHE": {
            "seq": np.stack(seq),
            "scal": np.stack(scal),
            "mask": np.stack(mask),
            "tgt": np.stack(tgt),
            "rot": np.stack(rot),
            "spd": np.array(spd, np.float32),
            "traj": np.array(traj),
            "real": np.array(real),
            "labels": labels,
        },
    }


def predict_private(
    ns: dict[str, object],
    model_dir: Path,
    eval_paths: list[str],
    stats: dict[str, np.ndarray],
    n_each: int,
) -> np.ndarray:
    device = ns["DEVICE"]
    build_features_clean = ns["build_features_clean"]
    normalize = ns["normalize"]
    seqs, scals, rots, bases = [], [], [], []
    for p in eval_paths:
        x = pd.read_csv(p)[["x", "y", "z"]].to_numpy()
        seq, sc22, rot, base, _ = build_features_clean(x)
        seq_n, sc_n = normalize(seq, sc22, stats)
        seqs.append(seq_n)
        scals.append(sc_n)
        rots.append(rot)
        bases.append(base)
    seq_t = torch.tensor(np.stack(seqs))
    scal_t = torch.tensor(np.stack(scals))
    rot_t = np.stack(rots)
    base_t = np.stack(bases)
    mask_t = torch.ones(len(seq_t), 11)
    flip_t = torch.tensor(ns["Y_FLIP"], device=device)
    x_raw_t = torch.tensor(np.stack([ns["load_sample"](p) for p in eval_paths]))

    def predict_resid(path: Path, factory) -> np.ndarray:
        model = factory().to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model_state"])
        model.eval()
        out = []
        with torch.no_grad():
            for start in range(0, len(seq_t), 256):
                s = seq_t[start : start + 256].to(device)
                c = scal_t[start : start + 256].to(device)
                m = mask_t[start : start + 256].to(device)
                pr = model(s, c, m).cpu().numpy()
                sf = s.clone()
                sf[:, :, flip_t] *= -1
                pf = model(sf, c, m).cpu().numpy()
                pf[:, 1] *= -1
                out.append((pr + pf) / 2)
        r = np.concatenate(out)
        return base_t + np.einsum("bij,bj->bi", rot_t.transpose(0, 2, 1), r)

    def predict_h(path: Path) -> np.ndarray:
        model = ns["HyperPhysics_xy2"]().to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model_state"])
        model.eval()

        def forward(z: torch.Tensor) -> np.ndarray:
            out = []
            with torch.no_grad():
                for start in range(0, len(z), 256):
                    batch = z[start : start + 256].to(device)
                    ft, df, pl, th, _, _, _, rt, sp, _, _ = model.get_features(
                        batch, model.mean_stats, model.std_stats
                    )
                    pred, _, _ = model(ft, df, pl, th, sp, rt)
                    out.append(pred.cpu().numpy())
            return np.concatenate(out)

        pr = forward(x_raw_t)
        xf = x_raw_t.clone()
        xf[:, :, 1] *= -1
        pf = forward(xf)
        pf[:, 1] *= -1
        return (pr + pf) / 2

    preds = []
    for k in range(n_each):
        preds.append(predict_resid(model_dir / f"phaseG_full_{k}.pt", ns["AttnGRU"]))
    for k in range(n_each):
        preds.append(predict_resid(model_dir / f"phaseODE_full_{k}.pt", ns["ODEModel"]))
    for k in range(n_each):
        preds.append(predict_h(model_dir / f"phaseH_full_{k}.pt"))
    return np.mean(preds, axis=0)


def run_setting(
    ns: dict[str, object],
    setting: str,
    mode: str,
    ids: list[str],
    paths: list[str],
    x_all: np.ndarray,
    y_all: np.ndarray,
    remove_mask: np.ndarray | None,
    args: argparse.Namespace,
) -> dict[str, object]:
    folds = np.array([stable_fold_id(sample_id, args.folds) for sample_id in ids])
    val_mask = folds == args.val_fold
    train_mask = ~val_mask
    if remove_mask is not None:
        train_mask &= ~remove_mask

    setting_dir = Path(args.out_dir) / setting
    model_dir = setting_dir / "models"
    setting_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    train_paths = np.asarray(paths, dtype=object)[train_mask].astype(str).tolist()
    val_paths = np.asarray(paths, dtype=object)[val_mask].astype(str).tolist()
    train_y = y_all[train_mask]
    val_y = y_all[val_mask]
    train_x = x_all[train_mask]

    built = build_stats_and_cache(ns, train_paths, train_y)
    ns["CACHE"] = built["CACHE"]
    ns["GRU_ODE_EPOCHS"] = int(args.gru_ode_epochs)
    ns["H_EPOCHS"] = int(args.h_epochs)
    ns["N_EACH"] = int(args.n_each)
    ns["MODELS_DIR"] = str(model_dir)
    ns["DEVICE"] = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else ("cpu" if args.device == "auto" else args.device))

    start_time = time.time()
    for k in range(args.n_each):
        path = model_dir / f"phaseG_full_{k}.pt"
        if args.resume and path.exists():
            continue
        torch.save({"model_state": ns["train_cache_seed"](k, ns["AttnGRU"])}, path)
    for k in range(args.n_each):
        path = model_dir / f"phaseODE_full_{k}.pt"
        if args.resume and path.exists():
            continue
        torch.save({"model_state": ns["train_cache_seed"](k, ns["ODEModel"])}, path)
    for k in range(args.n_each):
        path = model_dir / f"phaseH_full_{k}.pt"
        if args.resume and path.exists():
            continue
        torch.save({"model_state": ns["train_h_seed"](k, train_x, train_y)}, path)

    pred = predict_private(ns, model_dir, val_paths, built["STATS"], args.n_each)
    m = metrics(pred, val_y)
    pd.DataFrame({"id": np.asarray(ids, dtype=object)[val_mask], "x": pred[:, 0], "y": pred[:, 1], "z": pred[:, 2]}).to_csv(
        setting_dir / "validation_predictions.csv", index=False
    )
    result = {
        "setting": setting,
        "noise_mode": mode,
        "device": str(ns["DEVICE"]),
        "n_each": int(args.n_each),
        "gru_ode_epochs": int(args.gru_ode_epochs),
        "h_epochs": int(args.h_epochs),
        "folds": int(args.folds),
        "val_fold": int(args.val_fold),
        "train_rows": int(train_mask.sum()),
        "validation_rows": int(val_mask.sum()),
        "removed_total": int(remove_mask.sum()) if remove_mask is not None else 0,
        "removed_from_training_fold": int((remove_mask & ~val_mask).sum()) if remove_mask is not None else 0,
        "validation_policy": "keep_all_validation_rows",
        "test_policy": "flags_only_no_removal",
        **m,
        "elapsed_sec": round(time.time() - start_time, 2),
        "model_dir": str(model_dir),
    }
    (setting_dir / "metrics.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description="Private LB 1st noise-removal validation ablation")
    parser.add_argument("--data-dir", default=os.environ.get("DATA_DIR", "./Data"))
    parser.add_argument("--out-dir", default="private_lb_noise_ablation")
    parser.add_argument("--settings", nargs="*", default=["origin", "criteria_a_conservative", "criteria_b_strict"])
    parser.add_argument("--folds", type=int, default=5)
    parser.add_argument("--val-fold", type=int, default=0)
    parser.add_argument("--n-each", type=int, default=1, help="Use 10 for full Private LB recipe.")
    parser.add_argument("--gru-ode-epochs", type=int, default=8, help="Use 55 for full Private LB recipe.")
    parser.add_argument("--h-epochs", type=int, default=3, help="Use 12 for full Private LB recipe.")
    parser.add_argument("--device", default="auto")
    parser.add_argument("--resume", action="store_true")
    args = parser.parse_args()

    ns = load_private_namespace()
    ns["DEVICE"] = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else ("cpu" if args.device == "auto" else args.device))
    data_dir = Path(args.data_dir)
    train_dir = data_dir / "train"
    paths = sorted(glob.glob(str(train_dir / "*.csv")))
    ids = [Path(p).stem for p in paths]
    labels_df = pd.read_csv(data_dir / "train_labels.csv").sort_values("id").reset_index(drop=True)
    y_all = labels_df[["x", "y", "z"]].to_numpy(np.float32)
    if labels_df["id"].astype(str).tolist() != ids:
        raise ValueError("train file id order does not match sorted train_labels.csv")
    x_all = np.stack([ns["load_sample"](p) for p in paths]).astype(np.float32)

    out_dir = Path(args.out_dir)
    train_quality, mask_a, mask_b = build_noise_reports(data_dir, out_dir, ids, x_all, y_all)
    print(
        "QUALITY",
        json.dumps(
            {
                "train_total": len(ids),
                "criteria_a_removed": int(mask_a.sum()),
                "criteria_b_removed": int(mask_b.sum()),
                "device": str(ns["DEVICE"]),
            },
            ensure_ascii=False,
        ),
        flush=True,
    )

    remove_masks = {
        "origin": None,
        "criteria_a_conservative": mask_a,
        "criteria_b_strict": mask_b,
    }
    results = []
    for setting in args.settings:
        if setting not in remove_masks:
            raise ValueError(f"unknown setting: {setting}")
        print("RUN_SETTING", setting, flush=True)
        results.append(run_setting(ns, setting, setting, ids, paths, x_all, y_all, remove_masks[setting], args))
        print("RESULT", json.dumps(results[-1], ensure_ascii=False), flush=True)

    df = pd.DataFrame(results)
    csv_path = out_dir / "private_lb_noise_ablation_results.csv"
    df.to_csv(csv_path, index=False)
    (out_dir / "private_lb_noise_ablation_results.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("SAVED", csv_path)



## 4. Use Notebook Namespace

The script runner normally imports model definitions from the original notebook. Here we already loaded them above, so the runner uses this notebook namespace.


In [ ]:
def load_private_namespace() -> dict[str, object]:
    return globals()


## 5. Run Origin / Criteria A / Criteria B


In [ ]:
import sys

sys.argv = [
    "private_lb_noise_ablation_runner.py",
    "--data-dir", "./Data",
    "--out-dir", "private_lb_noise_ablation_gpu_n1_e8",
    "--n-each", "1",
    "--gru-ode-epochs", "8",
    "--h-epochs", "3",
    "--settings", "origin", "criteria_a_conservative", "criteria_b_strict",
    "--resume",
]
main()


## 6. Result Summary


In [ ]:
import pandas as pd

result_path = Path("private_lb_noise_ablation_gpu_n1_e8/private_lb_noise_ablation_results.csv")
df = pd.read_csv(result_path)
cols = [
    "setting", "device", "n_each", "gru_ode_epochs", "h_epochs",
    "train_rows", "validation_rows", "removed_from_training_fold",
    "hit", "hits", "mean", "median", "q90", "elapsed_sec",
]
display(df[cols].sort_values("hit", ascending=False))
